In [1]:
import os
import glob
import time
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
from shapely.geometry import Polygon
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from pyproj import Transformer

# ------------------- Sliding window -------------------
def slide_geotiff(geotiff_path, window_size, stride):
    tiles = []
    with rasterio.open(geotiff_path) as src:
        for y in range(0, src.height - window_size + 1, stride):
            for x in range(0, src.width - window_size + 1, stride):
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)
                new_transform = from_origin(
                    src.transform.c + x * src.transform.a,
                    src.transform.f + y * src.transform.e,
                    src.transform.a,
                    -src.transform.e
                )
                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "offset_xy": (x, y),
                    "tif_image": os.path.basename(geotiff_path)
                })
    return tiles

# ------------------- Inference overlay -------------------
def overlay_segmentation_on_array(results, transform, offset_xy):
    object_data = []
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data

    for mask, box, cls, conf in zip(results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
        if conf < 0.5:
            continue
        mask_points = np.array(mask.tolist(), dtype=np.float32)
        if len(mask_points) < 3:
            continue
        mask_gis_coords = [rasterio.transform.xy(transform, y, x) for x, y in mask_points]
        cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
        lon_center, lat_center = rasterio.transform.xy(transform, cy, cx)
        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "mask_points_gis": json.dumps(mask_gis_coords),
            "confidence": float(conf),
            "class": int(cls),
            "tile_name": offset_xy,
            "bbox_x1": box[0].item(),
            "bbox_y1": box[1].item(),
            "bbox_x2": box[2].item(),
            "bbox_y2": box[3].item(),
        })
    return object_data

# ------------------- Mask IoU suppression -------------------
def mask_iou(poly1, poly2):
    if not poly1.is_valid or not poly2.is_valid:
        return 0
    inter = poly1.intersection(poly2).area
    union = poly1.union(poly2).area
    return inter / union if union != 0 else 0

def mask_iou_suppression(object_infos, iou_thresh=0.5):
    kept = []
    polys = [Polygon(json.loads(obj['mask_points_gis'])) for obj in object_infos]
    used = set()
    for i, p1 in enumerate(polys):
        if i in used:
            continue
        group = [object_infos[i]]
        used.add(i)
        for j in range(i+1, len(polys)):
            if j in used:
                continue
            if mask_iou(p1, polys[j]) > iou_thresh:
                group.append(object_infos[j])
                used.add(j)
        # Retain the most confident object in the group
        kept.append(max(group, key=lambda x: x['confidence']))
    return kept

# ------------------- DBSCAN + Mask IoU combined suppression -------------------
def global_suppression_pipeline_combined(object_infos, eps_meters=5.0):
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    centroids = []
    filtered_infos = []
    for obj in object_infos:
        poly = Polygon(json.loads(obj['mask_points_gis']))
        if poly.is_valid and not poly.is_empty:
            x, y = transformer.transform(poly.centroid.x, poly.centroid.y)
            centroids.append([x, y])
            filtered_infos.append(obj)
    if not centroids:
        return object_infos
    clustering = DBSCAN(eps=eps_meters, min_samples=1).fit(centroids)
    df = pd.DataFrame(filtered_infos)
    df['cluster'] = clustering.labels_
    final_infos = []
    for cid in df['cluster'].unique():
        group = df[df['cluster'] == cid].to_dict(orient='records')
        refined_group = mask_iou_suppression(group, iou_thresh=0.5)
        final_infos.extend(refined_group)
    return final_infos

# ------------------- Full inference runner -------------------
def run_inference(folder_path, model_path, output_csv, window_size=3000, stride=3000, batch_size=16):
    model = YOLO(model_path)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    all_object_infos = []

    for tif_image in tqdm(tif_images, desc="Processing Images"):
        slices = slide_geotiff(tif_image, window_size, stride)
        slice_batches = [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]
        for batch in tqdm(slice_batches, desc=f"Processing {os.path.basename(tif_image)}", leave=False):
            pil_imgs = [Image.fromarray(np.moveaxis(s['img_array'][:3], 0, -1)) for s in batch]
            with torch.inference_mode():
                results_batch = model(pil_imgs, device=device, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                objects = overlay_segmentation_on_array(results, slice_data['transform'], slice_data['offset_xy'])
                all_object_infos.extend(objects)
        torch.cuda.empty_cache()
        gc.collect()

    # 💡 Combined DBSCAN + Mask IoU suppression
    print("\n🔍 Applying Combined DBSCAN + Mask IoU Suppression...")
    suppressed_infos = global_suppression_pipeline_combined(all_object_infos, eps_meters=5.0)
    pd.DataFrame(suppressed_infos).to_csv(output_csv, index=False)
    print(f"\n✅ Suppressed results saved to {output_csv}")

